# Lab Tùy chọn: Hàm Chi phí (Cost Function)
<figure>
    <center> <img src="./images/C1_W1_L3_S2_Lecture_b.png"  style="width:1000px;height:200px;" ></center>
</figure>



## Mục tiêu
Trong lab này bạn sẽ:
- triển khai và khám phá hàm `cost` (chi phí) cho hồi quy tuyến tính với một biến. 


## Công cụ
Trong lab này chúng ta sẽ sử dụng: 
- NumPy, một thư viện phổ biến cho tính toán khoa học
- Matplotlib, một thư viện phổ biến để vẽ đồ thị
- các hàm vẽ đồ thị cục bộ trong file lab_utils_uni.py ở thư mục hiện tại

In [ ]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from lab_utils_uni import plt_intuition, plt_stationary, plt_update_onclick, soup_bowl
plt.style.use('./deeplearning.mplstyle')

## Phát biểu bài toán

Bạn muốn có một mô hình có thể dự đoán giá nhà dựa trên diện tích của căn nhà.  
Hãy sử dụng hai điểm dữ liệu giống như lab trước - một căn nhà 1000 feet vuông được bán với giá \\$300,000 và một căn nhà 2000 feet vuông được bán với giá \\$500,000.


| Diện tích (1000 sqft)     | Giá (1000 đô la) |
| -------------------| ------------------------ |
| 1                 | 300                      |
| 2                  | 500                      |


In [ ]:
x_train = np.array([1.0, 2.0])           #(diện tích tính bằng 1000 feet vuông)
y_train = np.array([300.0, 500.0])           #(giá tính bằng 1000 đô la)

## Tính toán Chi phí (Cost)
Thuật ngữ 'cost' (chi phí) trong bài này có thể hơi gây nhầm lẫn vì dữ liệu là về giá nhà. Ở đây, cost là thước đo cho việc mô hình của chúng ta dự đoán giá mục tiêu của căn nhà tốt như thế nào. Thuật ngữ 'price' (giá) được dùng cho dữ liệu về nhà.

Phương trình cho cost với một biến là:
  $$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2 \tag{1}$$ 
 
trong đó 
  $$f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{2}$$
  
- $f_{w,b}(x^{(i)})$ là dự đoán của chúng ta cho ví dụ $i$ sử dụng các tham số $w,b$.  
- $(f_{w,b}(x^{(i)}) -y^{(i)})^2$ là bình phương của hiệu giữa giá trị mục tiêu và dự đoán.   
- Các hiệu này được cộng dồn trên tất cả $m$ ví dụ và chia cho `2m` để tạo ra cost, $J(w,b)$.  
>Lưu ý, trong bài giảng phạm vi tổng thường từ 1 đến m, trong khi mã nguồn sẽ từ 0 đến m-1.


Mã bên dưới tính cost bằng cách lặp qua từng ví dụ. Trong mỗi vòng lặp:
- `f_wb`, một giá trị dự đoán được tính
- hiệu giữa giá trị mục tiêu và dự đoán được tính và bình phương.
- giá trị này được cộng vào tổng cost.

In [ ]:
def compute_cost(x, y, w, b): 
    """
    Tính hàm chi phí (cost) cho hồi quy tuyến tính.
    
    Args:
      x (ndarray (m,)): Dữ liệu, m ví dụ 
      y (ndarray (m,)): giá trị mục tiêu
      w,b (scalar)    : tham số mô hình  
    
    Returns
        total_cost (float): Cost khi sử dụng w,b làm tham số cho hồi quy tuyến tính
               để khớp với các điểm dữ liệu trong x và y
    """
    # số lượng ví dụ huấn luyện
    m = x.shape[0] 
    
    cost_sum = 0 
    for i in range(m): 
        f_wb = w * x[i] + b   
        cost = (f_wb - y[i]) ** 2  
        cost_sum = cost_sum + cost  
    total_cost = (1 / (2 * m)) * cost_sum  

    return total_cost

## Trực giác về Hàm Chi phí

<img align="left" src="./images/C1_W1_Lab02_GoalOfRegression.PNG"    style=" width:380px; padding: 10px;  " /> Mục tiêu của bạn là tìm một mô hình $f_{w,b}(x) = wx + b$, với các tham số $w,b$, có thể dự đoán chính xác giá trị nhà dựa trên đầu vào $x$. Cost là thước đo cho việc mô hình chính xác đến đâu trên dữ liệu huấn luyện.

Phương trình cost (1) ở trên cho thấy nếu $w$ và $b$ có thể được chọn sao cho các dự đoán $f_{w,b}(x)$ khớp với dữ liệu mục tiêu $y$, thì số hạng $(f_{w,b}(x^{(i)}) - y^{(i)})^2 $ sẽ bằng 0 và cost được tối thiểu hóa. Trong ví dụ đơn giản với hai điểm này, bạn có thể đạt được điều đó!

Trong lab trước, bạn đã xác định rằng $b=100$ cho ra một nghiệm tối ưu, vì vậy hãy đặt $b$ bằng 100 và tập trung vào $w$.

<br/>
Bên dưới, sử dụng thanh trượt để chọn giá trị của $w$ sao cho cost là nhỏ nhất. Đồ thị có thể mất vài giây để cập nhật.

In [ ]:
plt_intuition(x_train,y_train)

Đồ thị chứa một vài điểm đáng chú ý.
- cost là nhỏ nhất khi $w = 200$, khớp với kết quả từ lab trước
- Vì hiệu giữa giá trị mục tiêu và dự đoán được bình phương trong phương trình cost, cost tăng nhanh chóng khi $w$ quá lớn hoặc quá nhỏ.
- Sử dụng `w` và `b` được chọn bằng cách tối thiểu hóa cost cho ra một đường thẳng khớp hoàn hảo với dữ liệu.

## Trực quan hóa Hàm Chi phí - 3D

Bạn có thể thấy cost thay đổi như thế nào theo *cả* `w` và `b` bằng cách vẽ đồ thị 3D hoặc sử dụng đồ thị đường đồng mức (contour plot).   
Đáng chú ý là một số phần vẽ đồ thị trong khóa học này có thể khá phức tạp. Các hàm vẽ đồ thị đã được cung cấp sẵn và mặc dù việc đọc qua mã để làm quen với các phương pháp có thể hữu ích, điều này không cần thiết để hoàn thành khóa học thành công. Các hàm này nằm trong file lab_utils_uni.py ở thư mục hiện tại.

### Tập dữ liệu lớn hơn
Việc xem xét một kịch bản với nhiều điểm dữ liệu hơn là rất hữu ích. Tập dữ liệu này bao gồm các điểm dữ liệu không nằm trên cùng một đường thẳng. Điều đó có ý nghĩa gì đối với phương trình cost? Chúng ta có thể tìm $w$ và $b$ sao cho cost bằng 0 không? 

In [ ]:
x_train = np.array([1.0, 1.7, 2.0, 2.5, 3.0, 3.2])
y_train = np.array([250, 300, 480,  430,   630, 730,])

Trong đồ thị đường đồng mức, nhấp vào một điểm để chọn `w` và `b` nhằm đạt được cost thấp nhất. Sử dụng các đường đồng mức để hướng dẫn lựa chọn của bạn. Lưu ý, có thể mất vài giây để cập nhật đồ thị. 

In [ ]:
plt.close('all') 
fig, ax, dyn_items = plt_stationary(x_train, y_train)
updater = plt_update_onclick(fig, ax, x_train, y_train, dyn_items)

Ở trên, hãy chú ý các đường nét đứt trong đồ thị bên trái. Chúng biểu thị phần cost đóng góp bởi mỗi ví dụ trong tập huấn luyện của bạn. Trong trường hợp này, các giá trị khoảng $w=209$ và $b=2.4$ cho cost thấp nhất. Lưu ý rằng, vì các ví dụ huấn luyện của chúng ta không nằm trên một đường thẳng, cost nhỏ nhất không bằng 0.

### Bề mặt Chi phí Lồi (Convex)
Việc hàm cost bình phương giá trị mất mát (loss) đảm bảo rằng 'bề mặt lỗi' (error surface) lồi giống như một cái tô súp. Nó luôn có một điểm cực tiểu có thể đạt được bằng cách đi theo gradient trên tất cả các chiều. Trong đồ thị trước, vì các chiều $w$ và $b$ có tỷ lệ khác nhau, điều này không dễ nhận ra. Đồ thị sau, trong đó $w$ và $b$ đối xứng, đã được trình bày trong bài giảng:

In [ ]:
soup_bowl()

# Chúc mừng!
Bạn đã học được những điều sau:
 - Phương trình cost cung cấp một thước đo cho việc các dự đoán của bạn khớp với dữ liệu huấn luyện tốt như thế nào.
 - Việc tối thiểu hóa cost có thể cho ra các giá trị tối ưu của $w$, $b$.